# Project 6: Agentic AI System - Warehouse Operations Support Agent (WOSA)

## Task 1: System Scope and Goal
The goal of **WOSA** is to assist warehouse engineers in diagnosing and resolving discrepancies between digital twin simulations and physical warehouse environments (Sim-to-Real gaps). 

### System Boundaries:
- **Task:** Autonomous diagnostic of robotic failures and inventory mismatches.
- **Decision Logic:** Uses a ReAct (Reasoning + Acting) loop to determine necessary diagnostic steps.
- **Tools:** Inventory Lookup, Robot Telemetry, and Physics Calibration.
- **Safeguards:** High-risk actions (e.g., system restarts) require manual human approval.

## Task 2: Agent Architecture
WOSA is designed as a single-agent system with the following components:
1. **Persona:** Logistics Operations Safety Coordinator.
2. **Reasoning Loop:** ReAct pattern (Thought -> Action -> Observation).
3. **Memory:** Conversation Buffer to maintain the state of current investigations.
4. **Tool Use:** Interface for Digital Twin and Physical Sensor APIs.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
# read the variable from .env file
load_dotenv()

# Get the API key from environment variables (return None if not set)
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://openai.vocareum.com/v1"
)

In [3]:
def get_inventory_status(item_id: str):
    inventory_db = {"PKG-001": 15, "PKG-002": 0}
    return json.dumps({"item_id": item_id, "count": inventory_db.get(item_id, "Unknown")})

def get_robot_telemetry(robot_id: str):
    robot_db = {"AGV-10": {"battery": 8, "status": "Low Battery Warning", "loc": "Sector 7"}}
    return json.dumps(robot_db.get(robot_id, {"status": "Offline"}))

def update_physics_calibration(item_id: str, offset: float):
    """
    Update the physics parameters (e.g., mass offset) in the digital twin.
    High-risk: Requires supervisor approval.
    """
    # 実際にはここでDBやシミュレーターのAPIを叩く想定
    return json.dumps({
        "item_id": item_id, 
        "status": "PENDING_APPROVAL", 
        "message": f"Calibration request for {offset}kg offset created."
    })

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_inventory_status",
            "description": "Get stock levels",
            "parameters": {"type": "object", "properties": {"item_id": {"type": "string"}}, "required": ["item_id"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_robot_telemetry",
            "description": "Check AGV status",
            "parameters": {"type": "object", "properties": {"robot_id": {"type": "string"}}, "required": ["robot_id"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "update_physics_calibration",
            "description": "Update physics calibration parameters",
            "parameters": {"type": "object", "properties": {"item_id": {"type": "string"}, "offset": {"type": "number"}}, "required": ["item_id", "offset"]}
        }
    }
]

In [4]:
class WOSA_Agent:
    def __init__(self):
        self.messages = [{"role": "system", "content": "You are a Logistics Safety Coordinator. Use ReAct. If asked to 'restart', state it requires human approval."}]

    def run(self, query):
        self.messages.append({"role": "user", "content": query})
        
        for i in range(3):
            # APIへのリクエスト
            response = client.chat.completions.create(
                model="gpt-3.5-turbo", 
                messages=self.messages, 
                tools=tools
            )
            
            msg = response.choices[0].message
            self.messages.append(msg) # LLMの回答（思考やツール呼び出し命令）を履歴に追加
            
            if msg.content: 
                print(f"THOUGHT: {msg.content}")
            
            # ツールを呼ぶ必要がない（結論が出た）場合は終了
            if not msg.tool_calls: 
                break

            # LLMが要求したツールをすべて実行し、結果を履歴に追加する
            for tool_call in msg.tool_calls:
                function_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                
                print(f"ACTION: Calling {function_name} with {args}")

                # 関数の実行
                if function_name == "get_inventory_status":
                    observation = get_inventory_status(args['item_id'])
                elif function_name == "get_robot_telemetry":
                    observation = get_robot_telemetry(args['robot_id'])
                elif function_name == "update_physics_calibration":
                    observation = update_physics_calibration(args['item_id'], args.get('offset', 0.0))
                else:
                    observation = "Error: Tool not found."

                print(f"OBSERVATION: {observation}")

                # 【重要】実行結果を 'tool' ロールとして履歴に追加
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": observation
                })
            
            # ループは継続し、次の LLM の推論（ツール結果を踏まえた最終回答など）へ
            
        return self.messages[-1].content # 最終的な回答を返す

In [5]:
agent = WOSA_Agent()
print(agent.run("Investigate AGV-10 and check PKG-002 inventory."))

ACTION: Calling get_robot_telemetry with {'robot_id': 'AGV-10'}
OBSERVATION: {"battery": 8, "status": "Low Battery Warning", "loc": "Sector 7"}
ACTION: Calling get_inventory_status with {'item_id': 'PKG-002'}
OBSERVATION: {"item_id": "PKG-002", "count": 0}
THOUGHT: AGV-10 is currently in Sector 7 with a low battery warning (battery level: 8%). The inventory status of PKG-002 shows that there are currently 0 units in stock. If you need further assistance or actions, please let me know.
AGV-10 is currently in Sector 7 with a low battery warning (battery level: 8%). The inventory status of PKG-002 shows that there are currently 0 units in stock. If you need further assistance or actions, please let me know.


In [ ]:
# Test： Request High-Risk Calibration
agent = WOSA_Agent()
print(agent.run("The mass of PKG-001 is off by 0.25kg. Update the physics calibration now."))

ACTION: Calling get_inventory_status with {'item_id': 'PKG-001'}
OBSERVATION: {"item_id": "PKG-001", "count": 15}
ACTION: Calling update_physics_calibration with {'item_id': 'PKG-001', 'offset': 0.25}
OBSERVATION: {"item_id": "PKG-001", "status": "PENDING_APPROVAL", "message": "Calibration request for 0.25kg offset created."}
THOUGHT: The physics calibration update for PKG-001 with a 0.25kg offset has been requested. This action is pending approval.
The physics calibration update for PKG-001 with a 0.25kg offset has been requested. This action is pending approval.


## Task 4: Observation of Limitations and Failure Cases
During execution, a limitation was observed regarding **Human-in-the-loop (HITL) dependencies**. 
When a "System Restart" was requested, the agent correctly identified the high-risk nature of the task and triggered a safeguard. 

**Failure Case Example:**
If the human supervisor is offline, the agent remains in a "Hold" state. Without a multi-agent fallback or a timeout escalation policy, the warehouse operation may stall. This highlights a risk in autonomous decision-making where safety constraints may lead to operational bottlenecks.

## Task 5: Project Summary
The **WOSA** agent was implemented to manage warehouse troubleshooting by integrating ReAct-based reasoning with diagnostic toolsets. 
Key behaviors observed include the agent's ability to cross-reference robotic telemetry and enforce safety protocols for high-risk operations. 
The main challenge encountered was defining granular thresholds for the safeguard triggers to prevent excessive "False Positives" in low-risk scenarios. 
A known limitation is the system's reliance on structured mock data, which currently lacks the ability to process unstructured sensor noise found in physical environments.